In [1]:
import numpy as np
from tdmclient import ClientAsync
import cv2

from navigation_control import *
from computer_vision.vision import Vision 



ModuleNotFoundError: No module named 'computer_vision'

In [ ]:
vision = Vision()
#img = vision.load("initial_state.png")

        # 1) Ouvre la caméra et capture UNE image de référence
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la webcam.")

ret, frame0 = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("Impossible de capturer une frame webcam.")

cap.release()  # on ferme direct, l’image est stockée dans frame0

img = frame0

vision.initialize(img)  # couleurs + blur robot + canny + polygones (et fige dans static_polys)

# récupère les obstacles figés par initialize()
obstacles, robot_state = vision.process(
    img,
    show_debug=False,    # on ne veut pas l'image debug ici
    skip_blur=False,
    log_pose=False
)

robot = vision.detect_robot(img)

start = robot["center"], robot["angle"]

In [ ]:
global_path = ...  # from global planner
ekf = ...          #  EKF object

In [ ]:
async def run():
    client = ClientAsync()
    node = await client.wait_for_node()
    await node.lock()
    await node.wait_for_variables({"prox.horizontal"})

    # Pose:
    pose = [0.0, 0.0, 0.0]  # [x, y, theta] in WORLD

    print("Starting path following with obstacle avoidance...")
    for step in range(200):
        # --- get sensors from Thymio ---
        sensor_vals = list(node.v.prox.horizontal)

        # --- update local grid ---
        grid.update_from_sensor_vals(sensor_vals)

        # --- compute virtual goal ---
        virt_world, LA_world, F_att, F_rep = navigator.compute_virtual_goal(
            pose, global_path, sensor_vals=sensor_vals
        )
        if virt_world is None:
            break

        # --- controller: pose -> motors ---
        uL, uR, info = g2g.compute_motor_commands(pose, virt_world)

        await node.set_variables({
            "motor.left.target":  [uL],
            "motor.right.target": [uR],
        })

        # pose from ekf 
        pose = ekf.get_pose()

        # small sleep
        await client.sleep(0.1)

    # stop motors at the end
    await node.set_variables({
        "motor.left.target":  [0],
        "motor.right.target": [0],
    })
    await node.unlock()